# Environment report: daily, weekly and monthly

Where environmental issues are reported across Indonesia, and which issues they are. Built with
`great.viz.environment`; pick the report with `REPORT_TYPE`.

| `REPORT_TYPE` | Function | Layout |
|---|---|---|
| `'Daily'` | `env_daily()` | map, then issue bars above bars per island |
| `'Weekly'` | `env_weekly()` | map, issue bars, and a province pie when the top issue dominates |
| `'Monthly'` | `env_monthly()` | map across the top; issue bars, the pie, and bars per island underneath |

**One-time setup.** The map needs the `[geo]` extra (geopandas, shapely, requests):

```
py -3.11 -m pip install -e "C:\Users\ASUS\Documents\Research Project\GREAT Big Data Analysis Program[all]"
```

The 38-province GeoJSON is downloaded on first use and cached in your user directory.

In [ ]:
# Pick up edits to the great/ source without restarting the kernel.
%load_ext autoreload
%autoreload 2

### **Inputing the Requirement**

In [ ]:
REPORT_TYPE     = 'Weekly' # 'Daily', 'Weekly' or 'Monthly'
SOURCE          = 'https://docs.google.com/spreadsheets/d/119b-k0DDykSpZJ-BAmIANWKnlfYLBfho/edit?gid=137764228#gid=137764228' # shared as "Anyone with the link"
DOMINANCE_RATIO = 2.0      # Weekly and Monthly: top issue >= this x the second -> it leaves the bars and becomes a pie

### **Notebook Set Up**

In [ ]:
import re

import pandas as pd

from great import classify_issue, resolve_frame, validate_export
from great.viz import prep
from great.viz.environment import dominant_issue, env_daily, env_monthly, env_weekly, island_counts, province_counts

### **Accessing File**

Only the file id in `SOURCE` matters; it becomes Google's `export?format=xlsx` URL, which pandas
reads directly. `validate_export()` fails here if a column is missing or a Sentiment/Media value
is outside the canonical vocabulary.

In [ ]:
sheet_id = re.search(r'/spreadsheets/d/([\w-]+)', SOURCE).group(1)
df = pd.read_excel(f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx', header=1)
df = validate_export(df)

df1, start_date, end_date = prep.prepare_data(df)
print(f'{len(df1):,} rows | {start_date} - {end_date}')

### **Deriving `Isu_Inti` and `Provinsi`**

`classify_issue()` assigns the issue from the keyword rules in `great/issues.py`. `resolve_frame()`
finds the province in the headline and mention text, and uses the `Location` column only when the
text has no match. In this export `Location` is where the *author* is, not where the event is
happening, so the text is the better signal. It also returns `Pulau` for the island bars.

In [ ]:
df1['Isu_Inti'] = df1['Mentions'].map(classify_issue)
df1[['Provinsi', 'Pulau']] = resolve_frame(df1)

counts = df1['Provinsi'].value_counts()
unresolved = counts.get('Tidak Terdeteksi', 0) + counts.get('Provinsi Tidak Spesifik', 0)
print(f'resolved to a province: {len(df1) - unresolved:,} of {len(df1):,} '
      f'({(len(df1) - unresolved) / len(df1):.1%})')
counts.head(10)

In [ ]:
df1['Isu_Inti'].value_counts()

### **Visualize**

Both reports drop rows classified "Noise/Tidak Relevan" before counting.

Every map looks the same: provinces shaded by post count, with a same-size badge showing the
count.

**Weekly and Monthly.** When the top issue has at least `DOMINANCE_RATIO` times the posts of the second
(e.g. 5,727 vs 717, 8x), one bar would flatten all the others. That issue is taken out of the
bar chart and shown as a pie of its provinces instead: the top nine, the rest as "Lainnya", with
"Provinsi Tidak Spesifik" as its own wedge. Below the ratio there is no pie.

**Daily.** The map, then issue bars above bars per island. No pie.

**Monthly.** The map spans the top. Below it are the issue bars, the pie (when there is one)
and the bars per island.

In [ ]:
if REPORT_TYPE == 'Daily':
    fig = env_daily(df1, start_date=start_date, end_date=end_date)
elif REPORT_TYPE == 'Weekly':
    fig = env_weekly(df1, start_date=start_date, end_date=end_date, dominance_ratio=DOMINANCE_RATIO)
else:
    fig = env_monthly(df1, start_date=start_date, end_date=end_date, dominance_ratio=DOMINANCE_RATIO)

### **The numbers behind the figure**

In [ ]:
report = df1[df1['Isu_Inti'] != 'Noise/Tidak Relevan']   # the reports drop noise the same way

print('pie issue:', dominant_issue(report['Isu_Inti'].value_counts(), DOMINANCE_RATIO))
print(island_counts(report).sort_values(ascending=False))

province_counts(report).sort_values('jumlah_unggahan', ascending=False).head(10)